<a href="https://colab.research.google.com/github/nanmodi/Blogsite/blob/main/text_genration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [34]:
from huggingface_hub import notebook_login

notebook_login()

In [6]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

In [7]:
eli5 = eli5.train_test_split(test_size=0.2)

In [11]:
eli5


DatasetDict({
    train: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers', 'title_urls', 'selftext_urls'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers', 'title_urls', 'selftext_urls'],
        num_rows: 1000
    })
})

In [14]:
eli5['train'][0]

{'q_id': '5q94qe',
 'title': 'Why are many trades jobs, such as sanitation and street sweeping, unionized and highly paid?',
 'selftext': 'URL_0 I saw this comic and it made me think. The street sweeper would have a highly paid job when it is a government job, that is given decent wages by politicians. Why are many trade jobs more highly paid than traditional university degree required employment, such as teaching?',
 'category': 'Economics',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dcxh364'],
  'text': ["While a comic like this may have an element of truth, don't take it too literally. Here's a [job analysis]( URL_0 ) for garbage collectors showing a median salary of $33,800. The same site's listing [for teachers]( URL_1 ) shows a median salary of $54,890."],
  'score': [6],
  'text_urls': [['http://money.usnews.com/careers/best-jobs/garbage-collector/salary',
    'http://money.usnews.com/careers/best-jobs/elementary-school-teacher']]},
 'title_urls': ['url'],
 'selft

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '5q94qe',
 'title': 'Why are many trades jobs, such as sanitation and street sweeping, unionized and highly paid?',
 'selftext': 'URL_0 I saw this comic and it made me think. The street sweeper would have a highly paid job when it is a government job, that is given decent wages by politicians. Why are many trade jobs more highly paid than traditional university degree required employment, such as teaching?',
 'category': 'Economics',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dcxh364'],
 'answers.text': ["While a comic like this may have an element of truth, don't take it too literally. Here's a [job analysis]( URL_0 ) for garbage collectors showing a median salary of $33,800. The same site's listing [for teachers]( URL_1 ) shows a median salary of $54,890."],
 'answers.score': [6],
 'answers.text_urls': [['http://money.usnews.com/careers/best-jobs/garbage-collector/salary',
   'http://money.usnews.com/careers/best-jobs/elementary-school-teacher']],
 'title_urls': [

In [17]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [19]:
preprocess_function(eli5['train'][:2])

{'input_ids': [[3633, 257, 9048, 588, 428, 743, 423, 281, 5002, 286, 3872, 11, 836, 470, 1011, 340, 1165, 7360, 13, 3423, 338, 257, 685, 21858, 3781, 16151, 10289, 62, 15, 1267, 329, 15413, 26668, 4478, 257, 14288, 9588, 286, 720, 2091, 11, 7410, 13, 383, 976, 2524, 338, 13487, 685, 1640, 7799, 16151, 10289, 62, 16, 1267, 2523, 257, 14288, 9588, 286, 720, 4051, 11, 23, 3829, 13], [40, 1101, 407, 6635, 1654, 281, 17226, 5194, 460, 4197, 287, 257, 4506, 7442, 13, 383, 18197, 4523, 3777, 314, 1101, 3910, 286, 340, 262, 370, 4051, 290, 345, 561, 761, 257, 1263, 736, 2353, 13, 632, 373, 546, 838, 11111, 416, 1315, 8589, 290, 3463, 546, 2026, 8059, 13, 314, 1101, 1654, 356, 460, 787, 257, 4833, 530, 11, 475, 788, 345, 923, 284, 423, 2761, 43539, 340, 13, 4380, 481, 766, 326, 262, 3516, 318, 9648, 284, 1745, 262, 4506, 7442, 351, 530, 1021, 357, 361, 326, 338, 772, 1744, 737, 843, 611, 262, 43539, 318, 2793, 11, 788, 339, 3516, 481, 2192, 923, 284, 905, 1051, 286, 11881, 22475, 379, 617, 966,

In [20]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1496 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1091 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (3304 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1049 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1446 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1314 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1204 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2683 > 1024). Running this sequence through the model will result in indexing errors


In [22]:
tokenized_eli5['train'][0]

{'input_ids': [3633,
  257,
  9048,
  588,
  428,
  743,
  423,
  281,
  5002,
  286,
  3872,
  11,
  836,
  470,
  1011,
  340,
  1165,
  7360,
  13,
  3423,
  338,
  257,
  685,
  21858,
  3781,
  16151,
  10289,
  62,
  15,
  1267,
  329,
  15413,
  26668,
  4478,
  257,
  14288,
  9588,
  286,
  720,
  2091,
  11,
  7410,
  13,
  383,
  976,
  2524,
  338,
  13487,
  685,
  1640,
  7799,
  16151,
  10289,
  62,
  16,
  1267,
  2523,
  257,
  14288,
  9588,
  286,
  720,
  4051,
  11,
  23,
  3829,
  13],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [23]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [24]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [25]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [26]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [29]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.919543,3.776553
2,3.831094,3.766585
3,3.790121,3.764500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3966, training_loss=3.8525447874314485, metrics={'train_runtime': 1164.837, 'train_samples_per_second': 27.236, 'train_steps_per_second': 3.405, 'total_flos': 1036204926566400.0, 'train_loss': 3.8525447874314485, 'epoch': 3.0})

In [30]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 43.14
